# 面试问题：给定训练算力、数据库存与上线目标，怎样用 Scaling Law 选择 LLM 参数量和训练 Token？

        ## 可直接复述的回答主线

        1. Scaling Law 是预算内的实验规划工具，不是把参数越做越大的口号。
2. 先统一参数量、训练 Token、有效算力和模型状态显存的单位，再比较候选方案。
3. 朴素地选择最大模型会忽略数据量不足，常出现参数多但损失更高的欠训练模型。
4. 底层估算可用训练 FLOP 约等于六倍参数量乘 Token 数，并把模型项与数据项分别展示。
5. 选型必须同时满足算力、清洗后数据库存、显存和上线延迟约束。
6. 最终结论要由小规模拟合实验校准，不能把教学公式直接当作线上质量承诺。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例是一个中文客服模型团队在固定 GPU 周预算内选择预训练方案。六个候选方案使用脱敏的容量规划字段：参数量、训练 Token、可用清洗数据和单卡显存需求。数据为结构与真实评审表一致的离线教学样本，不代表任何厂商模型的真实损失。

In [1]:
import math  # 引入向上取整与有限数检查所需的数学函数。
plans = [{"name": "1.3B-800B", "params_b": 1.3, "tokens_b": 800, "state_gb": 21}, {"name": "3B-400B", "params_b": 3.0, "tokens_b": 400, "state_gb": 48}, {"name": "7B-180B", "params_b": 7.0, "tokens_b": 180, "state_gb": 112}, {"name": "13B-100B", "params_b": 13.0, "tokens_b": 100, "state_gb": 208}, {"name": "20B-60B", "params_b": 20.0, "tokens_b": 60, "state_gb": 320}, {"name": "34B-35B", "params_b": 34.0, "tokens_b": 35, "state_gb": 544}]  # 定义六个具有参数、数据和显存语义的训练候选。
compute_budget_flops = 8.0e21  # 设定本轮预训练允许消耗的有效计算预算。
clean_token_inventory_b = 80.0  # 记录去重和质量过滤后真正可用的 Token 库存。
print("教学实验输入：中文客服模型候选规划")  # 输出案例名称，避免把结果误认为线上基准。
print("方案          参数/B   Token/B   模型状态/GB")  # 输出容量评审表的字段标题。
for plan in plans:  # 逐条展示六个候选而不是只留下变量。
    print(f"{plan['name']:<12} {plan['params_b']:>6.1f} {plan['tokens_b']:>9.0f} {plan['state_gb']:>12.0f}")  # 输出当前候选的可读配置。
print(f"有效算力预算={compute_budget_flops:.2e} FLOP，清洗数据库存={clean_token_inventory_b:.0f}B Token")  # 输出两项硬约束供后续对照。

教学实验输入：中文客服模型候选规划
方案          参数/B   Token/B   模型状态/GB
1.3B-800B       1.3       800           21
3B-400B         3.0       400           48
7B-180B         7.0       180          112
13B-100B       13.0       100          208
20B-60B        20.0        60          320
34B-35B        34.0        35          544
有效算力预算=8.00e+21 FLOP，清洗数据库存=80B Token


## 2. Baseline / 基线：直接选择预算内参数最大的模型

最朴素方案只检查训练 FLOP，然后在可运行候选中选择参数量最大的模型。它很容易解释，却没有检查数据是否足够，也没有比较模型项和数据项对预测损失的贡献。

In [2]:
def training_flops(plan):  # 用统一单位估算当前候选的一次预训练计算量。
    return 6.0 * plan["params_b"] * plan["tokens_b"] * 1.0e18  # 按六倍参数量乘 Token 数返回 FLOP。
compute_feasible = [plan for plan in plans if training_flops(plan) <= compute_budget_flops]  # 仅按计算预算筛选朴素候选。
baseline = max(compute_feasible, key=lambda plan: plan["params_b"])  # 选择参数量最大的候选作为朴素基线。
baseline_data_gap = max(0.0, baseline["tokens_b"] - clean_token_inventory_b)  # 计算基线超出清洗数据库存的缺口。
print("Baseline 选择结果")  # 标记当前输出属于朴素策略。
print(f"选择={baseline['name']}，训练FLOP={training_flops(baseline):.2e}")  # 展示基线选择与预算占用。
print(f"需要Token={baseline['tokens_b']:.0f}B，数据缺口={baseline_data_gap:.0f}B")  # 暴露基线忽略数据约束的后果。

Baseline 选择结果
选择=34B-35B，训练FLOP=7.14e+21
需要Token=35B，数据缺口=0B


## 3. 底层实现：拆开模型项、数据项与计算约束

教学公式使用两个幂律项模拟参数扩张和数据扩张的边际收益。绝对数值没有生产含义，重要的是在同一假设下观察每项贡献，并把数据库存作为硬约束。

In [3]:
def scaling_components(plan):  # 计算候选的模型项、数据项和总预测损失。
    model_term = 0.70 * plan["params_b"] ** -0.34  # 估算扩大参数量带来的可约损失项。
    data_term = 0.95 * plan["tokens_b"] ** -0.28  # 估算增加训练 Token 带来的可约损失项。
    predicted_loss = 1.45 + model_term + data_term  # 加上不可约项得到同口径预测损失。
    return model_term, data_term, predicted_loss  # 返回可解释的三项而非单一总分。
evaluated = []  # 收集所有计算预算内候选的分项结果。
for plan in compute_feasible:  # 对同一批候选执行统一 Scaling Law 估算。
    model_term, data_term, predicted_loss = scaling_components(plan)  # 取得当前候选的两个贡献项。
    evaluated.append({"plan": plan, "model_term": model_term, "data_term": data_term, "loss": predicted_loss})  # 保存当前候选的可审计结果。
print("中间分项：模型项越小表示参数更充分，数据项越小表示训练更充分")  # 解释下表每列的方向。
print("方案          模型项    数据项    预测损失   数据可行")  # 输出底层估算表头。
for row in evaluated:  # 逐条输出模型项与数据项以解释权衡。
    data_ok = row["plan"]["tokens_b"] <= clean_token_inventory_b  # 判断当前方案是否能由清洗数据支撑。
    print(f"{row['plan']['name']:<12} {row['model_term']:>7.4f} {row['data_term']:>8.4f} {row['loss']:>9.4f} {str(data_ok):>8}")  # 输出分项和硬约束状态。

中间分项：模型项越小表示参数更充分，数据项越小表示训练更充分
方案          模型项    数据项    预测损失   数据可行
1.3B-800B     0.6403   0.1462    2.2364    False
3B-400B       0.4818   0.1775    2.1093    False
7B-180B       0.3612   0.2219    2.0332    False
13B-100B      0.2927   0.2617    2.0043    False
20B-60B       0.2528   0.3019    2.0047     True
34B-35B       0.2111   0.3511    2.0121     True


## 4. 结果表与结果解读

先看忽略数据约束时的理论最优，再看加入数据库存后的可交付方案。损失差异只用于解释权衡；真正选型还需小模型扫描、吞吐实测和下游任务评测。

In [4]:
unconstrained_best = min(evaluated, key=lambda row: row["loss"])  # 找到只受计算预算约束的理论最优。
data_feasible = [row for row in evaluated if row["plan"]["tokens_b"] <= clean_token_inventory_b]  # 加入清洗数据库存硬约束。
corrected_best = min(data_feasible, key=lambda row: row["loss"])  # 在真实可用数据范围内选择损失最低方案。
comparison = [("最大参数基线", baseline["name"], scaling_components(baseline)[2], baseline["tokens_b"] <= clean_token_inventory_b), ("仅算力最优", unconstrained_best["plan"]["name"], unconstrained_best["loss"], unconstrained_best["plan"]["tokens_b"] <= clean_token_inventory_b), ("算力+数据修正", corrected_best["plan"]["name"], corrected_best["loss"], True)]  # 构造三种决策口径的同指标对照。
print("策略              选择          预测损失   数据可交付")  # 输出最终对照表表头。
for strategy, name, loss, data_ok in comparison:  # 展示每种策略的选择与约束结果。
    print(f"{strategy:<17} {name:<12} {loss:>8.4f} {str(data_ok):>10}")  # 输出同一指标下的可比较结果。
print(f"解读：数据约束使选择从 {unconstrained_best['plan']['name']} 调整为 {corrected_best['plan']['name']}，避免重复低质量语料。")  # 给出读者无需猜测的结论。

策略              选择          预测损失   数据可交付
最大参数基线            34B-35B        2.0121       True
仅算力最优             13B-100B       2.0043      False
算力+数据修正           20B-60B        2.0047       True
解读：数据约束使选择从 13B-100B 调整为 20B-60B，避免重复低质量语料。


## 5. 失败案例与修正

失败案例是把未清洗的 100B Token 当成可用库存，导致计划隐含 20B Token 缺口。修正不是简单缩小模型，而是把数据库存门禁前置，再重新比较同一损失指标。

In [5]:
failed_plan = unconstrained_best["plan"]  # 取出忽略数据库存时会被推荐的候选。
repeated_or_low_quality_b = max(0.0, failed_plan["tokens_b"] - clean_token_inventory_b)  # 估算被迫重复或降质的数据量。
corrected_plan = corrected_best["plan"]  # 取出加入数据门禁后的可交付候选。
print(f"错误行为：推荐 {failed_plan['name']}，其中 {repeated_or_low_quality_b:.0f}B Token 没有清洗库存支撑。")  # 直接展示失败规模。
print(f"修正行为：推荐 {corrected_plan['name']}，Token 需求 {corrected_plan['tokens_b']:.0f}B 不超过库存。")  # 展示修正后满足的硬约束。

错误行为：推荐 13B-100B，其中 20B Token 没有清洗库存支撑。
修正行为：推荐 20B-60B，Token 需求 60B 不超过库存。


## 6. 生产边界

线上规划应以预跑实验重新拟合指数和常数，并加入有效吞吐、故障重启、数据混合、许可证、推理延迟和目标任务质量。这里没有模拟通信、优化器状态或训练失败，预测损失也不能外推到任意架构。

In [6]:
production_checks = {"小规模拟合": False, "数据许可证": True, "集群有效吞吐实测": False, "下游任务评测": False, "推理SLO评测": False}  # 列出教学公式之外仍需完成的生产检查。
pending_checks = [name for name, passed in production_checks.items() if not passed]  # 汇总尚未由本实验覆盖的评审项。
print("生产交接清单：", "、".join(pending_checks))  # 明确输出不能直接支持上线承诺的原因。

生产交接清单： 小规模拟合、集群有效吞吐实测、下游任务评测、推理SLO评测


## 7. 最小回归测试

只保护计算预算、数据门禁和选择结果三类关键不变量，不再用断言代替教学输出。

In [7]:
assert len(plans) >= 5  # 保证真实案例至少包含五个可比较候选。
assert all(training_flops(plan) <= compute_budget_flops for plan in compute_feasible)  # 保证进入比较表的候选都满足计算预算。
assert corrected_plan["tokens_b"] <= clean_token_inventory_b  # 保证修正选择不依赖不存在的清洗数据。
assert math.isfinite(corrected_best["loss"])  # 保证最终预测损失为可解释的有限数。